# Clinical Decision Support Governance Evidence Package

**Use case:** diagnosis-assistance risk review for breast-cancer morphology records.  
**Regulatory lens:** FDA clinical decision support transparency, HIPAA privacy safeguards, clinical safety monitoring, and audit-ready model evidence.


## 1. Intended Use and Governance Boundary

The model is treated as a **clinical decision support aid**. It prioritizes cases for clinician review and provides transparent evidence about which morphology patterns increased or decreased the malignant-risk estimate. It is not positioned as autonomous diagnosis, final triage authority, or a substitute for pathology review.

The benchmark dataset contains no direct identifiers and no patient demographics. This limits HIPAA and protected-class fairness conclusions. The notebook therefore separates **clinical morphology-strata safety review** from **protected-class bias review** and avoids claiming demographic fairness that the data cannot support.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.calibration import calibration_curve
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, brier_score_loss, classification_report, confusion_matrix,
    precision_recall_fscore_support, roc_auc_score, roc_curve
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
OUTPUT_DIR = Path("nb08_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
DATA_PATH = Path("nb08_healthcare_breast_cancer_data.csv")

## 2. Dataset Export and Data Lineage

The analysis uses the Wisconsin Diagnostic Breast Cancer benchmark. The target is recoded so that `diagnosis_malignant = 1` represents the clinically higher-risk class.

In [ ]:
# Export the exact benchmark dataset used by the notebook.
bunch = load_breast_cancer(as_frame=True)
df = bunch.frame.copy().rename(columns={"target": "diagnosis_original_sklearn_target"})
df.insert(0, "patient_id", [f"BCDR-{i:04d}" for i in range(len(df))])
df["diagnosis_malignant"] = (df["diagnosis_original_sklearn_target"] == 0).astype(int)
df["diagnosis_label"] = df["diagnosis_malignant"].map({1: "malignant", 0: "benign"})
df.to_csv(DATA_PATH, index=False)

feature_cols = [c for c in df.columns if c not in [
    "patient_id", "diagnosis_original_sklearn_target", "diagnosis_malignant", "diagnosis_label"
]]
X = df[feature_cols].copy()
y = df["diagnosis_malignant"].astype(int)

pd.DataFrame({
    "evidence_item": ["records", "features", "malignant prevalence", "missing values", "direct identifiers"],
    "value": [len(df), len(feature_cols), round(float(y.mean()), 3), int(X.isna().sum().sum()), "none in benchmark export"]
})

## 3. Data Quality and Privacy Controls

The benchmark contains measurement fields only. The review still records the minimum governance checks expected before model development: record count, feature count, missingness, target prevalence, and identifier status. In a production EHR deployment, this section would be expanded to include PHI mapping, access control evidence, extract approval, and retention policy references.

In [ ]:
quality = pd.DataFrame({
    "feature": X.columns,
    "missing_count": X.isna().sum().values,
    "min": X.min().round(3).values,
    "median": X.median().round(3).values,
    "max": X.max().round(3).values,
})
quality.head(12)

## 4. Model Development

The preferred repository execution path uses `hugiml-core` through `HUGIMLClassifierNative`. A transparent sklearn fallback is included only so the notebook remains executable in environments where the package has not yet been installed. The governance analysis structure remains the same: holdout validation, calibration, patient-level evidence, morphology-strata safety review, and monitoring indicators.

In [ ]:
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)

try:
    from hugiml import HUGIMLClassifierNative
    model = HUGIMLClassifierNative(
        B=10, L=1, G=1e-4, topK=120,
        adaptive_binning=True,
        b_candidates=[3, 5, 7, 10, 15],
        origColumns=X_train.columns.tolist(),
    )
    c=model.fit(X_train, y_train)
    model_name = "hugiml-core HUGIMLClassifierNative"
except Exception as exc:
    warnings.warn(f"Using transparent sklearn fallback because hugiml-core is unavailable: {exc}")
    model = Pipeline([
        ("scale", StandardScaler()),
        ("lr", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE)),
    ])
    model.fit(X_train, y_train)
    model_name = "transparent sklearn logistic fallback"

model_name

In [ ]:
def predict_probability(model, X):
    proba = model.predict_proba(X)
    return proba[:, 1] if getattr(proba, "ndim", 1) == 2 else proba

def expected_calibration_error(y_true, y_prob, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ids = np.digitize(y_prob, bins[1:-1], right=True)
    ece = 0.0
    for b in range(n_bins):
        mask = ids == b
        if np.any(mask):
            ece += mask.mean() * abs(y_true[mask].mean() - y_prob[mask].mean())
    return float(ece)

def choose_safety_threshold(y_true, y_prob, min_sensitivity=0.95):
    best_threshold, best_specificity = 0.5, -1.0
    for threshold in np.linspace(0.05, 0.95, 181):
        pred = (y_prob >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
        sensitivity = tp / max(tp + fn, 1)
        specificity = tn / max(tn + fp, 1)
        if sensitivity >= min_sensitivity and specificity > best_specificity:
            best_threshold, best_specificity = threshold, specificity
    return float(best_threshold)

p_train = predict_probability(model, X_train)
p_holdout = predict_probability(model, X_holdout)
threshold = choose_safety_threshold(y_holdout, p_holdout, min_sensitivity=0.95)
pred_holdout = (p_holdout >= threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_holdout, pred_holdout, labels=[0, 1]).ravel()

summary = pd.DataFrame({
    "metric": ["ROC-AUC", "Sensitivity", "Specificity", "Accuracy", "Brier score", "ECE", "Safety threshold"],
    "value": [
        roc_auc_score(y_holdout, p_holdout),
        tp / max(tp + fn, 1),
        tn / max(tn + fp, 1),
        accuracy_score(y_holdout, pred_holdout),
        brier_score_loss(y_holdout, p_holdout),
        expected_calibration_error(y_holdout.to_numpy(), p_holdout, 10),
        threshold,
    ]
})
summary.assign(value=lambda d: d.value.round(4))

## 5. Clinical Validation Evidence

The operating point is selected with a safety constraint: high malignant-case sensitivity is prioritized over raw accuracy. For audit review, this makes the threshold rationale explicit and makes the resulting false-positive burden visible.

In [ ]:
fpr, tpr, _ = roc_curve(y_holdout, p_holdout)
prob_true, prob_pred = calibration_curve(y_holdout, p_holdout, n_bins=8, strategy="quantile")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(fpr, tpr, label=f"ROC-AUC = {roc_auc_score(y_holdout, p_holdout):.3f}")
axes[0].plot([0, 1], [0, 1], linestyle="--", linewidth=1)
axes[0].set_title("Discrimination")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("Sensitivity")
axes[0].legend(loc="lower right")

axes[1].plot(prob_pred, prob_true, marker="o")
axes[1].plot([0, 1], [0, 1], linestyle="--", linewidth=1)
axes[1].set_title("Calibration")
axes[1].set_xlabel("Mean predicted malignant risk")
axes[1].set_ylabel("Observed malignant rate")
plt.tight_layout()
plt.show()

## 6. Explainability and Case-Level Review

In [ ]:
def local_contributions(model, X_row):
    if isinstance(model, Pipeline):
        scaler = model.named_steps["scale"]
        lr = model.named_steps["lr"]
        values = scaler.transform(pd.DataFrame([X_row]))[0]
        coefs = lr.coef_[0]
        contrib = values * coefs
        out = pd.DataFrame({
            "feature": X_row.index,
            "standardized_value": values,
            "coefficient": coefs,
            "log_odds_contribution": contrib,
        })
        return out.assign(abs_contribution=lambda d: d.log_odds_contribution.abs()).sort_values("abs_contribution", ascending=False)
    try:
        # HUGIML plotting utilities can expose active patterns for case-level explanations.
        from hugiml.plots import HUGPlotter
        return pd.DataFrame({"note": ["Use HUGPlotter(model).plot_active_patterns(X_holdout, sample_idx=...) for active HUG-pattern review."]})
    except Exception:
        return pd.DataFrame({"note": ["Active HUG-pattern extraction not available in this runtime."]})

case_pos = int(np.argmax(p_holdout))
case = X_holdout.iloc[case_pos]
case_risk = p_holdout[case_pos]
case_truth = y_holdout.iloc[case_pos]
contrib = local_contributions(model, case).head(10)
contrib

In [ ]:
if "log_odds_contribution" in contrib.columns:
    c = contrib.sort_values("log_odds_contribution")
    fig, ax = plt.subplots(figsize=(9, 4.8))
    ax.barh(c["feature"], c["log_odds_contribution"])
    ax.axvline(0, linewidth=1)
    ax.set_title(f"Representative Case Review — predicted malignant risk {case_risk:.3f}")
    ax.set_xlabel("Contribution toward malignant-risk score")
    plt.tight_layout()
    plt.show()

## 7. Morphology-Strata Safety Review

The dataset does not contain age, sex, race, payer status, or other demographic attributes. Therefore, this is not a protected-class fairness assessment. It is a clinical safety review by morphology strata to identify where review burden or errors concentrate.

In [ ]:
def make_morphology_strata(X):
    compactness = pd.qcut(X["mean compactness"], 3, labels=["low compactness", "mid compactness", "high compactness"])
    radius = pd.qcut(X["mean radius"], 3, labels=["small radius", "mid radius", "large radius"])
    return radius.astype(str) + " / " + compactness.astype(str)

def subgroup_review(X, y, p, threshold):
    groups = make_morphology_strata(X)
    pred = (p >= threshold).astype(int)
    rows = []
    for group in sorted(groups.unique()):
        mask = groups == group
        if mask.sum() < 10:
            continue
        tn, fp, fn, tp = confusion_matrix(y[mask], pred[mask], labels=[0, 1]).ravel()
        rows.append({
            "stratum": group,
            "n": int(mask.sum()),
            "malignant_rate": y[mask].mean(),
            "sensitivity": tp / max(tp + fn, 1),
            "specificity": tn / max(tn + fp, 1),
            "false_negative_count": int(fn),
            "false_positive_count": int(fp),
            "mean_predicted_risk": float(np.mean(p[mask])),
        })
    return pd.DataFrame(rows).sort_values(["false_negative_count", "mean_predicted_risk"], ascending=[False, False])

strata = subgroup_review(X_holdout, y_holdout, p_holdout, threshold)
strata.round(3)

## 8. Monitoring Evidence

Monitoring focuses on two questions: whether the distribution of model scores has shifted, and whether review workload is still consistent with the safety threshold. The dashboard uses these outputs to state operational implications, rather than showing a monitoring graphic detached from the data.

In [ ]:
def psi(expected, observed, bins=10):
    edges = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0
    exp_counts, _ = np.histogram(expected, bins=edges)
    obs_counts, _ = np.histogram(observed, bins=edges)
    exp_pct = np.maximum(exp_counts / max(exp_counts.sum(), 1), 1e-6)
    obs_pct = np.maximum(obs_counts / max(obs_counts.sum(), 1), 1e-6)
    return float(np.sum((obs_pct - exp_pct) * np.log(obs_pct / exp_pct)))

review_queue = pd.DataFrame({
    "patient_id": df.loc[X_holdout.index, "patient_id"].values,
    "actual_label": y_holdout.map({1: "malignant", 0: "benign"}).values,
    "predicted_malignant_risk": p_holdout,
    "predicted_review_flag": pred_holdout.astype(bool),
})
review_queue["review_bucket"] = pd.cut(
    review_queue["predicted_malignant_risk"],
    bins=[0, 0.2, 0.5, 0.8, 1.0],
    labels=["low", "watch", "elevated", "critical"],
    include_lowest=True,
)

monitoring = pd.DataFrame({
    "indicator": ["train-vs-holdout PSI", "holdout review rate", "critical bucket count", "false negative count", "false positive count"],
    "value": [
        psi(pd.Series(p_train), pd.Series(p_holdout)),
        review_queue["predicted_review_flag"].mean(),
        int((review_queue["review_bucket"] == "critical").sum()),
        int(fn),
        int(fp),
    ]
})
monitoring

## 9. Audit Artifacts

The notebook writes a compact evidence package that can be attached to a governance review: summary metrics, morphology-strata safety review, representative patient explanation, and review queue output.

In [ ]:
summary_payload = {
    "model_name": model_name,
    "records": int(len(df)),
    "features": int(len(feature_cols)),
    "target_definition": "diagnosis_malignant = 1 for malignant records; benchmark target recoded from sklearn convention",
    "roc_auc": float(roc_auc_score(y_holdout, p_holdout)),
    "threshold": float(threshold),
    "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    "sensitivity": float(tp / max(tp + fn, 1)),
    "specificity": float(tn / max(tn + fp, 1)),
    "brier_score": float(brier_score_loss(y_holdout, p_holdout)),
    "ece_10_bin": float(expected_calibration_error(y_holdout.to_numpy(), p_holdout, 10)),
    "privacy_note": "benchmark export contains morphology measurements and synthetic patient_id only; no direct identifiers",
    "fairness_note": "protected-class fairness cannot be assessed because demographic attributes are not present",
}

(OUTPUT_DIR / "clinical_governance_summary.json").write_text(json.dumps(summary_payload, indent=2))
strata.to_csv(OUTPUT_DIR / "morphology_strata_safety_review.csv", index=False)
review_queue.to_csv(OUTPUT_DIR / "clinical_review_queue.csv", index=False)
if isinstance(contrib, pd.DataFrame):
    contrib.to_csv(OUTPUT_DIR / "representative_patient_explanation.csv", index=False)

summary_payload